In [1]:
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import cooler
import bioframe
import cooltools
from cooltools.lib import plotting
import cooler as clr
import h5py
import coolbox
from coolbox.api import *
from coolbox.utilities import GenomeRange
from pygenometracks import readBed, readGtf
from pygenometracks.tracks import BedTrack

import glob
from pathlib import Path


%cd /home/varshini/uc_analyses/
import utils, coolbox_helpers
%cd /home/varshini/LoopCall_Analysis_Scripts
import re

warnings.filterwarnings("ignore")

import logging
logging.getLogger('matplotlib.font_manager').disabled = True
from matplotlib.backends.backend_pdf import PdfPages

/home/varshini/uc_analyses
/home/varshini/LoopCall_Analysis_Scripts


In [3]:
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 12

plt.rcParams.update({
    "text.usetex": False})
plt.rc('pdf',fonttype = 42)

In [20]:
def make_region_plot_mod(region, resolution, microc_file,
                     bw_list, condition_order, gene_annot_name, bedgraph_fname=None, plot_variants=False,
                     file_bottom=None, cmap_chip='tab10', bigwig_bins=1600, autoscale=True, plot_bed=False,
                        variant_track_height=2, custom_valranges=None, plot_genes=True):
    
    """
    Function that makes a plot of a region with Micro-C, ChIP-seq, and Pro-seq.
    
    region: (str) string that specifies region for plotting.
    resolution: (int) resolution of micro-c map used
    highlight_region_list: List of regions of format [chr, start, end] to highlight
    microc_file: (str) file path to micro-c data
    microc_title: (str) title for micro-c
    chip_list_list: List of lists containing paths to bigwigs
    forward_proseq_list: List of paths to forward strand pro-seq files
    reverse_proseq_list: List of paths to reverse strand pro-seq files
    condition_order: List indicating order of conditions to plot. 
    """
    print(region, resolution, microc_file)
    heatmap = Cool(microc_file, file_bottom=file_bottom, resolution=resolution, **MICROC_PLOTTING_PARAMS) 
    frame = heatmap + Spacer(SPACER_HEIGHT*3)
    if plot_bed:
        frame+=NewBed(*BED_PARAMS) + TrackHeight(3) + Spacer(SPACER_HEIGHT)
    #frame +=  BED(finemapped_variant_name, border_only=True, 
    #              alpha=0.1, show_score='yes', linewidth=0.5, display="collapsed", height=1, edgecolor="#0099BB") + Spacer(SPACER_HEIGHT)
    if len(bw_list)>0:
        bigwig_list = coolbox_helpers.make_bigwig_list(bw_list, region, condition_order, autoscale=False)
        cmap_arr = sns.color_palette(cmap_chip, n_colors=len(bigwig_list))

        for i, bigwig in enumerate(bigwig_list):
            if autoscale:
                frame += coolbox_helpers.auto_scale_bigwig(bigwig, region) + Color(cmap_arr[i]) + Spacer(SPACER_HEIGHT)
            else:
                frame += bigwig + MaxValue(custom_valranges[i]) + MinValue(0) + Color(cmap_arr[i]) + Spacer(SPACER_HEIGHT)

    if bedgraph_fname is not None:
        frame += BigWig(bedgraph_fname) + Title("HbF_Base") + Color("#EE7733") + MaxValue(30) + MinValue(0)+ Spacer(SPACER_HEIGHT)
    
    if plot_variants:
        frame+=SNP(finemapped_variant_name, **VARIANTS_PARAMS) \
        + HLines(0, line_width=2, line_style='-',color='k') + Title("HbF GWAS [-log10(p)]")  + TrackHeight(variant_track_height) + Spacer(SPACER_HEIGHT)
    
    if plot_genes:
        annotations = NewBed(*GTF_PARAMS)
#     annotations = GTF(gene_annot_name, color=["#000000"], gene_rows=1, 
#                       filter_named_genes=True, row_filter=None) 
    frame+=annotations

    return frame


def multiplot(mcool1, mcool2, peaks, resolution=5000):
    for index, row in peaks.iterrows():
        plt.figure()
        region_str = get_locus_from_coord(row, flank=200_000)

        # Reverse these
        microc_file1 = mcool1
        microc_file2 = mcool2
        microc_title = f"{region_str} at {resolution}"
        heatmap = Cool(microc_file1, file_bottom=microc_file2,
                       resolution=resolution, **MICROC_PLOTTING_PARAMS) + Title(microc_title)
        frame = heatmap
        annotations = NewBed(*ANNOTATIONS_PARAMS)
        frame += annotations
        frame += XAxis()
        fig = frame.plot(region_str)



In [21]:
class ChromName(Track):
    """
    Track for show chromosome name.

    Parameters
    ----------
    fontsize : float
        Font name to show.

    offset : float
        Offset ratio to the start position.
    """

    DEFAULT_PROPERTIES = {
        "fontsize": 50,
        "offset": 0.45,
    }

    def __init__(self, **kwargs):
        properties = ChromName.DEFAULT_PROPERTIES.copy()
        properties.update(kwargs)
        super().__init__(properties)

    def fetch_data(self, gr: GenomeRange, **kwargs):
        return gr.chrom  # return chromosome name

    def plot(self, ax, gr: GenomeRange, **kwargs):
        x = gr.start + self.properties['offset'] * (gr.end - gr.start)
        ax.text(x, 0, gr.chrom, fontsize=self.properties['fontsize'])
        ax.set_xlim(gr.start, gr.end)


class NewBed(Track):
    """
    Custom pygenometracks class that allows for control over the aesthetics of gene annotations in
    region plots.
    """

    def __init__(self, coolbox_prop_dict, pygenometracks_prop_dict):
        super().__init__(coolbox_prop_dict)  # init
        self.pygenometracks_object = BedTrack(pygenometracks_prop_dict)

    def fetch_data(self, gr, **kwargs):
        pass

    def plot(self, ax, gr, **kwargs):
        #         x = gr.start + self.properties['offset'] * (gr.end - gr.start)
        #         ax.text(x, 0, gr.chrom, fontsize=self.properties['fontsize'])
        #         ax.set_xlim(gr.start, gr.end)
        self.pygenometracks_object.plot(ax, gr.chrom, gr.start, gr.end)
        ax.set_xlim([gr.start, gr.end])

In [22]:
# define the bigwig and RNAseq files 
base_dir = '/mnt/coldstorage/Varshmallow/'

def get_chip_list(base_dir='/mnt/coldstorage/Varshmallow/', phase=1, phase2=None, 
                  include=['RNASeq', 'ATAC', 'H3K27ac', 'H3K4me1']):

    chip_dict = {}
    chip_dict[f'RNASeq_P{phase}'] = f'{base_dir}Blood_RNAseq/Phase{phase}_RNA_chr.bw'
    
    # ATAC-Seq
    chip_dict[f'ATAC_P{phase}'] = f'{base_dir}Blood_ATAC/Phase{phase}_ATAC.bw'
    
    chip_dict['NIPBL'] = '/mnt/md0/varshini/Analysis/Blood/complete_analyses/bigwigs/GSM8864683_HUDEP-2_NIPBL_rep1_2_mean.bw'
 
    # GATA-1 
    
    
    if phase2 is not None:
        chip_dict[f'ATAC_P{phase2}'] = f'{base_dir}Blood_ATAC/Phase{phase2}_ATAC.bw'
        chip_dict[f'RNASeq_P{phase2}'] = f'{base_dir}Blood_RNAseq/Phase{phase2}_RNA_chr.bw'
        
    # Histone ChIP
    chip_dir = '/mnt/md0/varshini/Analysis/Blood/complete_analyses/bigwigs/'

    directory = os.fsencode(chip_dir)
    
    for file in os.listdir(directory):
        filename = os.fsdecode(file)
        rm = re.match(rf'\w*P{phase}_CTCF\S*', filename)
        if rm is not None:
            chip_dict[f'CTCF_P{phase}'] = f'{chip_dir}{rm.group(0)}'

        rm = re.match(rf'\w*P{phase}_H3K27ac\S*', filename)
        if rm is not None:
            chip_dict[f'H3K27ac_P{phase}']  = f'{chip_dir}{rm.group(0)}'

        rm = re.match(rf'\w*P{phase}_H3K4me1\S*', filename)
        if rm is not None:
            chip_dict[f'H3K4me1_P{phase}'] = f'{chip_dir}{rm.group(0)}'
       
        rm = re.match(r'\w*GATA1\S*', filename)
        if rm is not None:
            chip_dict[f'GATA1'] = f'{base_dir}Blood_ChIP/P{phase}/{rm.group(0)}'
            
        
    if phase2 is not None:
        for file in os.listdir(directory):
            filename = os.fsdecode(file)
            rm = re.match(rf'\w*P{phase2}_CTCF\S*', filename)
            if rm is not None:
                chip_dict[f'CTCF_P{phase2}'] = f'{chip_dir}{rm.group(0)}'

            rm = re.match(rf'\w*P{phase2}_H3K27ac\S*', filename)
            if rm is not None:
                chip_dict[f'H3K27ac_P{phase2}']  = f'{chip_dir}{rm.group(0)}'

            rm = re.match(rf'\w*P{phase2}_H3K4me1\S*', filename)
            if rm is not None:
                chip_dict[f'H3K4me1_P{phase2}'] = f'{chip_dir}{rm.group(0)}'

        
    chip_list = [chip_dict[key] for key in include]
    return chip_list


In [23]:
phases = ['Phase1', 'Phase2', 'Phase3']
reps = ['Rep1', 'Rep2']
uc_path = '/mnt/coldstorage/Varshmallow/Adipose_Blood_Merged/sankaran/rep_cools'
uc_filenames = [os.path.join(uc_path, f'Sankaran_{phase}_{rep}.250.mcool') for phase in phases for rep in reps]

savedir = os.path.join('/mnt/md0/varshini/Analysis/Blood/figs_v2/supp1')

In [24]:
GTF_FILENAME = "/mnt/md0/varshini/Gene_Annotations/refSeqSelect.bed"
GRNA = '/mnt/md0/varshini/Analysis/Blood/Tracks_Intersection_Analysis_NewCTCF/hbf_grna.bw'
finemapped_variant_name = '/mnt/md0/varshini/Analysis/Blood/hbf_gwas/METALOUT_inv1.bed'

# Define plotting aesthetics as global variables
HIGHLIGHT_PARAMS = {"color":"gray",
                   "alpha":0.5,
                   "border_line":False}

MICROC_PLOTTING_PARAMS = {"style":"matrix",
                          "transform":"log10",
                          "depth_ratio":"full",
                          "balance":True,
                          "cmap":"fall",
                          "max_value":-0.5,
                          "min_value":-4.5}

VARIANTS_PARAMS = {
    "col_chrom":0, 
    "col_pos":2,
    "col_pval":6, 
    "threshold":10e-8, 
    "min_value":0, 
    "size":20,
    "alpha":0.75}

SPACER_HEIGHT = 0.4

regions = {'compt_ex1': 'chr5:149023111-152003168',
        'compt_ex2': 'chr1:48000000-102000000',
        'compt_ex3': 'chr11:58000000-62000000',
        'globin_lcr': 'chr11:5007550-5509242',
        'loop_appear': 'chr5:173570169-174026097 ',
        'loop_disappear': 'chr7:11365292-12545212',
        'boundary_appear': 'chr8:41456263-41881802 ',
        #'boundary_disappear':'chr10:68928655-69146919',
        'boundary_disappear': 'chr8:75561811-79081548',
        'EE_loops': 'chr6:36967226-37358383'
    
}



In [25]:
uc_filenames

['/mnt/coldstorage/Varshmallow/Adipose_Blood_Merged/sankaran/rep_cools/Sankaran_Phase1_Rep1.250.mcool',
 '/mnt/coldstorage/Varshmallow/Adipose_Blood_Merged/sankaran/rep_cools/Sankaran_Phase1_Rep2.250.mcool',
 '/mnt/coldstorage/Varshmallow/Adipose_Blood_Merged/sankaran/rep_cools/Sankaran_Phase2_Rep1.250.mcool',
 '/mnt/coldstorage/Varshmallow/Adipose_Blood_Merged/sankaran/rep_cools/Sankaran_Phase2_Rep2.250.mcool',
 '/mnt/coldstorage/Varshmallow/Adipose_Blood_Merged/sankaran/rep_cools/Sankaran_Phase3_Rep1.250.mcool',
 '/mnt/coldstorage/Varshmallow/Adipose_Blood_Merged/sankaran/rep_cools/Sankaran_Phase3_Rep2.250.mcool']

In [26]:

region_str = regions["globin_lcr"]
resolution=3200

MICROC_PLOTTING_PARAMS = {"style":"matrix",
                          "transform":"log10",
                          "depth_ratio":"full",
                          "balance":True,
                          "cmap":"fall",
                          "max_value":-1,
                          "min_value":-4,
                         "fontsize":18}


GTF_PARAMS = [
     {"height":5},
    {"file":GTF_FILENAME,
     "max_labels":1000,
    #"gene_rows":3,
    "merge_transcripts":True,       #"gene_style":"normal",
    "border_only":"true",
     "color":"#000000"
    }] # PATH TO BED FILE OF GENE ANNOTATIONS

incl= []

chip_list = get_chip_list(include=incl, phase=1, phase2=3)


In [27]:

frame = make_region_plot_mod(region_str, resolution, uc_filenames[0],
                         chip_list, condition_order=incl, gene_annot_name=GTF_FILENAME,
                        bedgraph_fname=None, plot_variants=False, variant_track_height=5,
                         file_bottom = uc_filenames[1], cmap_chip="deep", autoscale=True, plot_bed=False)
fig = frame.plot(region_str)
fig.savefig(os.path.join(savedir,'globin_P1.svg'))

chr11:5007550-5509242 3200 /mnt/coldstorage/Varshmallow/Adipose_Blood_Merged/sankaran/rep_cools/Sankaran_Phase1_Rep1.250.mcool


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 22438/22438 [00:00<00:00, 33059.39it/s]
DEBUG:pygenometracks.tracks.GenomeTrack:ylim 7.9799999999999995,-0.08


You've reached the modified Cool class!
[-4 -2 -1]


In [28]:
frame = make_region_plot_mod(region_str, resolution, uc_filenames[2],
                         chip_list, condition_order=incl, gene_annot_name=GTF_FILENAME,
                        bedgraph_fname=None, plot_variants=False, variant_track_height=5,
                         file_bottom = uc_filenames[3], cmap_chip="deep", autoscale=True, plot_bed=False)
fig = frame.plot(region_str)
fig.savefig(os.path.join(savedir,'globin_P2.svg'))


chr11:5007550-5509242 3200 /mnt/coldstorage/Varshmallow/Adipose_Blood_Merged/sankaran/rep_cools/Sankaran_Phase2_Rep1.250.mcool


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 22438/22438 [00:00<00:00, 32835.20it/s]


You've reached the modified Cool class!
[-4 -2 -1]


DEBUG:pygenometracks.tracks.GenomeTrack:ylim 7.9799999999999995,-0.08


In [29]:
frame = make_region_plot_mod(region_str, resolution, uc_filenames[4],
                         chip_list, condition_order=incl, gene_annot_name=GTF_FILENAME,
                        bedgraph_fname=None, plot_variants=False, variant_track_height=5,
                         file_bottom = uc_filenames[5], cmap_chip="deep", autoscale=True, plot_bed=False)
fig = frame.plot(region_str)
fig.savefig(os.path.join(savedir,'globin_P3.svg'))

chr11:5007550-5509242 3200 /mnt/coldstorage/Varshmallow/Adipose_Blood_Merged/sankaran/rep_cools/Sankaran_Phase3_Rep1.250.mcool


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 22438/22438 [00:00<00:00, 32435.29it/s]
DEBUG:pygenometracks.tracks.GenomeTrack:ylim 7.9799999999999995,-0.08


You've reached the modified Cool class!
[-4 -2 -1]


In [30]:

region_str = regions["EE_loops"]
resolution=2000

incl= []

chip_list = get_chip_list(include=incl, phase=1, phase2=3)



In [31]:
frame = make_region_plot_mod(region_str, resolution, uc_filenames[0],
                         chip_list, condition_order=incl, gene_annot_name=GTF_FILENAME,
                        bedgraph_fname=None, plot_variants=False, variant_track_height=5,
                         file_bottom = uc_filenames[1], cmap_chip="deep", autoscale=True, plot_bed=False)
fig = frame.plot(region_str)
fig.savefig(os.path.join(savedir,'PIM_P1.svg'))



chr6:36967226-37358383 2000 /mnt/coldstorage/Varshmallow/Adipose_Blood_Merged/sankaran/rep_cools/Sankaran_Phase1_Rep1.250.mcool


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 22438/22438 [00:00<00:00, 32865.07it/s]
DEBUG:pygenometracks.tracks.GenomeTrack:ylim 5.68,-0.08


You've reached the modified Cool class!
[-4 -2 -1]


In [32]:
frame = make_region_plot_mod(region_str, resolution, uc_filenames[2],
                         chip_list, condition_order=incl, gene_annot_name=GTF_FILENAME,
                        bedgraph_fname=None, plot_variants=False, variant_track_height=5,
                         file_bottom = uc_filenames[3], cmap_chip="deep", autoscale=True, plot_bed=False)
fig = frame.plot(region_str)
fig.savefig(os.path.join(savedir,'PIM_P2.svg'))




chr6:36967226-37358383 2000 /mnt/coldstorage/Varshmallow/Adipose_Blood_Merged/sankaran/rep_cools/Sankaran_Phase2_Rep1.250.mcool


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 22438/22438 [00:01<00:00, 15448.29it/s]
DEBUG:pygenometracks.tracks.GenomeTrack:ylim 5.68,-0.08


You've reached the modified Cool class!
[-4 -2 -1]


In [33]:
frame = make_region_plot_mod(region_str, resolution, uc_filenames[4],
                         chip_list, condition_order=incl, gene_annot_name=GTF_FILENAME,
                        bedgraph_fname=None, plot_variants=False, variant_track_height=5,
                         file_bottom = uc_filenames[5], cmap_chip="deep", autoscale=True, plot_bed=False)
fig = frame.plot(region_str)
fig.savefig(os.path.join(savedir,'PIM_P3.svg'))


chr6:36967226-37358383 2000 /mnt/coldstorage/Varshmallow/Adipose_Blood_Merged/sankaran/rep_cools/Sankaran_Phase3_Rep1.250.mcool


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 22438/22438 [00:00<00:00, 32940.89it/s]
DEBUG:pygenometracks.tracks.GenomeTrack:ylim 5.68,-0.08


You've reached the modified Cool class!
[-4 -2 -1]
